# Tutorial 03 · How good is a design - before and after data collection?

The slides introduced the tools: the expected SE from the trials per cell, coverage (entropy) and the confound warning signs. Here you **use** them - on your own design and on the class data - and push them a bit further.

**Part A - before data collection**
1. Your draft's report
2. How many players would we need?
3. Break the metrics
4. Your draft vs. the predefined design and a random block

**Part B - after data collection**

5. The class estimates
6. Was σ right?
7. The price of control by analysis

**Part C - a black box with a known truth**

8. Your design vs. random, compared with the truth
9. Remove the repetitions in the analysis

Cells are named by their sequence; the last letter is the current trial: CI = incongruent after congruent.

## Setup

In [ ]:
import os
import sys

# the course folder is one level above tutorials/ (two levels above tutorials/solutions/)
root = os.path.abspath("..")
if not os.path.isdir(os.path.join(root, "resources")):
    root = os.path.abspath(os.path.join("..", ".."))

sys.path.insert(0, root)

In [ ]:
import numpy as np
import pandas as pd

from experiments.stroop import random_trials
from resources.stroop_data import load_stroop_data
from resources.design_quality import (cell_counts, expected_se, design_report, scoreboard, pool_sequences,
                                      trials_from_pattern, trial_sd, player_results, class_summary,
                                      repetition_check, example_class_data, black_box_study,
                                      plot_estimates, plot_black_box)

σ, the SD of a single trial, comes from the Session 1 data of the class (fallback: about 200 ms, a rough value from the literature).

In [ ]:
s1 = load_stroop_data(block="1")
n_s1 = 0
if not s1.empty:
    n_s1 = s1["participant"].nunique()

if n_s1 < 5:
    print("Fewer than 5 participants in the Session 1 data - using the literature value.")
    sigma = 200.0
else:
    sigma = trial_sd(s1)

print(f"SD of a single trial: sigma = {sigma:.0f} ms")

# Part A · Before data collection

## 1. Your draft's report

Load your group's draft from Tutorial 02. The report collects everything from the slides: trials per cell, expected SEs, coverage and the warning signs.

In [ ]:
GROUP = "example"   # your group name (the file designs/<group>.csv)

my_design = pd.read_csv(os.path.join(root, "designs", GROUP + ".csv"))
design_report(my_design, sigma).to_frame("your draft")

### Exercise 1
Check the expected SE of the sequence effect by hand: σ · √(1/n_CC + 1/n_CI + 1/n_IC + 1/n_II).

In [ ]:
counts = cell_counts(my_design, "sequence effect")

# ADD YOUR CODE HERE
counts["CC"], counts["CI"], counts["IC"], counts["II"]
se_by_hand = ...

print(se_by_hand)

## 2. How many players would we need?

The expected SE is per **player**. Averaging over n players divides it by √n: SE_class = SE_player / √n. (This ignores the differences between people - so it is a lower bound.)

### Exercise 2
1. How many players does the class need for a class SE of the sequence effect of **10 ms**? Solve SE_player / √n = 10 for n.
2. The sequence effect is about 30 ms. Is the class (about 25 players) large enough to see it?

In [ ]:
se_player = expected_se(cell_counts(my_design, "sequence effect"), "sequence effect", sigma)

# ADD YOUR CODE HERE
n_needed = ...

## 3. Break the metrics

The report gives numbers - but do they catch every bad design? `trials_from_pattern` builds a block from a string of C and I (random words and inks, no repetitions), e.g. `trials_from_pattern("CICI")`.

### Exercise 3
Build two bad designs with 96 trials each and compare their reports with your draft:
- **blocked**: 48 congruent trials, then 48 incongruent trials
- **runs of four**: CCCC IIII CCCC IIII ...

Which number reveals each problem - and which problem would the precision (SE) alone **not** reveal?

In [ ]:
# ADD YOUR CODE HERE
blocked = trials_from_pattern(...)
runs_of_four = trials_from_pattern(...)

scoreboard({"your draft": my_design, "blocked": blocked, "runs of four": runs_of_four}, sigma)

## 4. Your draft vs. the predefined design and a random block

These are the two blocks everybody plays: the **predefined design** (`designs/designed.csv` holds 100 SweetPea sequences of it - every player gets one at random; the table shows the first) and a **random block**.

In [ ]:
designed = pd.read_csv(os.path.join(root, "designs", "designed.csv"))

scoreboard({"your draft": my_design, "predefined design": pool_sequences(designed)[0],
            "random block": random_trials(seed=1)}, sigma)

Read out your SE of the sequence effect and your warning signs - we collect all groups on the board.

# Part B · After data collection

Everybody played the **predefined design** (block `designed`) and a **random block** (block `random`). With fewer than 5 players who played both, the notebook uses simulated stand-in data.

In [ ]:
data = load_stroop_data()
if not data.empty:
    data = data[data["block"].isin(["designed", "random"])]
players = data.groupby("participant")["block"].nunique() if not data.empty else pd.Series(dtype=int)

if (players == 2).sum() < 5:
    print("Fewer than 5 players with both blocks - using simulated stand-in data.")
    data = example_class_data(designed)

print(data["participant"].nunique(), "players")

## 5. The class estimates

For every player we estimate both effects in both blocks; the class estimate is the mean over players, its SE is SD(player estimates) / √n.

In [ ]:
stroop = player_results(data, "Stroop effect")
class_summary(stroop)

In [ ]:
sequence = player_results(data, "sequence effect")
class_summary(sequence)

In [ ]:
fig = plot_estimates({"Stroop effect": class_summary(stroop), "sequence effect": class_summary(sequence)})

## 6. Was σ right?

The predictions in Part A used σ from Session 1. Now we can measure σ in today's blocks.

### Exercise 4
1. Measure σ in today's data with `trial_sd(data)`.
2. Recompute the predicted class SE of the sequence effect for the predefined design: expected SE (with the new σ) / √n. Use the average expected SE over the 100 sequences of `designed`.
3. Compare with the observed class SE from Section 5. Which prediction is closer?

In [ ]:
n_players = len(sequence)

# ADD YOUR CODE HERE
sigma_today = ...
predicted = ...

## 7. The price of control by analysis

In the random blocks, repetitions of the ink colour can inflate the sequence effect. We can remove them **in the analysis** - or exclude them **by design**, as the predefined design does.

In [ ]:
check = repetition_check(data, "sequence effect", block="random")
check

In [ ]:
fig = plot_estimates({"sequence effect, random blocks": check})

### Exercise 5
1. How many trials per player does removing the repetitions cost?
2. If σ stays the same, the SE grows with √(trials before / trials after). Compute this factor and compare it with the observed SEs in the table.
3. What does excluding repetitions **by design** cost?

In [ ]:
# ADD YOUR CODE HERE
trials_before = check.loc["all trials", "trials used per player"]
trials_after = ...
factor = ...

# Part C · A black box with a known truth

## 8. Your design vs. random, compared with the truth

With real people we never know the true effects - so we cannot tell how close a design gets to them. A **simulation** can: here the players are simulated, and every simulated player has a true Stroop effect and a true sequence effect. How the simulation works stays hidden - treat it as a black box, like nature.

Choose what to compare with random sampling: the **predefined design** or **your own draft** (or both). `black_box_study` lets 500 simulated players play each design and compares every player's estimate with that player's true value:
- **bias**: the average deviation of the estimates from the truth
- **SD per player**: how much a single player's estimate scatters around the truth

The 500 simulated players are not a class - they are there to **measure the design**: the more simulated players, the more accurate these two numbers.

In [ ]:
designs_to_test = {
    "random": "random",                 # a fresh random block for every player
    "predefined": designed,             # one of the 100 SweetPea sequences per player
    "your draft": my_design,            # the same sequence for every player
}

truth = black_box_study(designs_to_test)
truth

The figure shows where the average of a class of 24 players typically lands, relative to the truth (dashed line): the dot is the bias, the bar spans ± 2 · SD / √24 - the range a class average scatters over. The next cells explain the formula.

In [ ]:
fig = plot_black_box(truth)

### From 500 simulated players to our class

In the course, about **24 people** play, and we report their **average**. How far is such a class average from the truth?
- **Scatter:** averaging 24 players shrinks it by √24: from SD to SD / √24.
- **Bias:** averaging does **not** shrink it - every player is off in the same direction.

Together, the class average deviates from the truth by the total error (root mean squared error):

$$\text{total error} = \sqrt{\text{bias}^2 + \text{SD}^2 / 24}$$

More players only shrink the second part. A bias stays, however many people play - only a better design removes it.

### Exercise 6
1. Compute the total error of a class of 24 for every design and effect. Which design gets closest to the truth - and is it because of the bias or because of the SD?
2. How large would the total error of the random block be for the sequence effect with **240** players instead of 24?

In [ ]:
# ADD YOUR CODE HERE
truth["total error, class of 24 (ms)"] = ...

truth

In [ ]:
# ADD YOUR CODE HERE
random_sequence = truth.loc[("sequence effect", "random")]
error_240 = ...

# fig = plot_black_box(truth, n_players=240)

**Questions:**
1. Add `trials_from_pattern("CCCCIIII" * 12)` (runs of four) to `designs_to_test`. How does the black box judge it - and how would real players react to it?
2. The black box only knows what was built into it. What can its verdict about designs **not** tell you?

## 9. Remove the repetitions in the analysis

In Part B (Section 7) you removed the trials that repeat the ink colour from the random blocks - **control by analysis**. With the black box we can now check what that does to the distance from the truth.

`"random, repetitions removed"` plays random blocks, but drops every trial whose ink colour repeats the previous one before estimating the effects.

In [ ]:
control = black_box_study({
    "random": "random",
    "random, repetitions removed": "random, repetitions removed",
    "predefined": designed,
})
control

In [ ]:
fig = plot_black_box(control)

### Exercise 7
1. Compute the total error of a class of 24 for the three designs, as in Exercise 6.
2. Removing the repetitions changes both parts of the total error. Which part gets smaller, which gets larger - and why?
3. Which is better: controlling the repetitions in the analysis or in the design?

In [ ]:
# ADD YOUR CODE HERE
control["total error, class of 24 (ms)"] = ...

control

## Take-away

- Your design's precision follows from its trials per cell - and the number of players decides what the class can see.
- Precision alone misses bad designs: coverage and the warning signs catch what the SE cannot.
- Predictions are only as good as σ - after the data, check them.
- Control by analysis costs trials, control by design costs nothing.
- A simulation knows the truth - but only the truth that was built into it.
- Removing confounds in the analysis trades bias for precision; removing them by design costs nothing.